In [1]:
#| default_exp rest

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [4]:
#| export
from rest.core import init_instance, process_seq
singleton, model_path = init_instance()

In [5]:
model_path = 'pelevin'

In [6]:
#| export
seq_length = 1024

model_path = f'./models/large/{model_path}'
from transformers import GPT2LMHeadModel,GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_path, pad_token_id = 50256)
model = GPT2LMHeadModel.from_pretrained(model_path).half()
model.config.pad_token_id = model.config.eos_token_id
model.cuda()
model.eval();

In [7]:
sum(p.numel() for p in model.parameters())

774030080

In [8]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    encoded_prompt = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt").cuda()
    encoded_prompt = encoded_prompt[:,length-(seq_length-1):]
    bad_words_ids = [tokenizer.encode('[')[0], tokenizer.encode('(')[0], tokenizer.encode('1\xa01')[1]]
    linebreak = tokenizer.encode("1\n1")[1]
    lb2 = tokenizer.encode("1 \n")[1]
    bad_words_ids += [] if allow_linebreak else [linebreak, lb2]
    bad_words_ids = [[b] for b in bad_words_ids] + [[linebreak,linebreak]]
    output_sequences = model.generate(
            input_ids=encoded_prompt,
            max_length=length + len(encoded_prompt[0]),
            temperature=1,
            top_k=0,
            top_p=0.9,
            do_sample=True,num_return_sequences=num_samples,
            bad_words_ids = bad_words_ids,
        )
    
    if len(output_sequences.shape) > 2:
            output_sequences.squeeze_()
    generated_sequences = []
    for generated_sequence_idx, generated_sequence in enumerate(output_sequences):
        generated_sequence = generated_sequence.tolist()
        text = tokenizer.decode(generated_sequence, clean_up_tokenization_spaces=True)
        total_sequence = text[len(tokenizer.decode(encoded_prompt[0], clean_up_tokenization_spaces=True)) :]
        generated_sequences.append(total_sequence)

    return process_seq(generated_sequences)

not setting adaptive thresholding
CPU times: user 13.4 s, sys: 329 ms, total: 13.8 s
Wall time: 1.04 s


[' Извини. Сама не знаю, - смутилась я. - Прости, я ошиблась. Что ты сказал? Как ты меня назвал? Я тебя чем-то обидела? Это как-то связано с тем... - Я помотала головой, вспоминая.',
 ' Федор Михайлович Достоевский, - отвечаю, и опять я не ошибся.',
 '... Нет, - сказал отец, - для этого у меня нету времени. Но я постараюсь. Это  таки да... Рама. Сними пиджак. Посвети вон туда. Да не в стену. В пол.',
 ' Имя ни к чему, у меня много имен. Ты должен понимать, что я не даю ложных обещаний, хотя в принципе могу это сделать. Именно поэтому тебе в моем повествовании будет несколько неловко.']

In [46]:
%%time
get_sample(' - ты кто?', 50, 4, False)

not setting adaptive thresholding
CPU times: user 13.5 s, sys: 313 ms, total: 13.8 s
Wall time: 1.02 s


[' Почему-то – он. Меня зовут Игорь Валерьевич. Хочу, чтобы ты знал, я не сумасшедший. Я действительно чего-то ищу и вот сейчас случайно нашёл.',
 ' Воин? Маг? Странник? Кто? Посмотри на мои запястья. Они связаны. Я из плена? Как тебе удалось меня освободить? Впрочем, не отвечай. Я не спрашиваю. Возможно, ты имеешь доступ к Глазу истины.',
 ' Впрочем, можешь не отвечать. Ты - Сосо. И продолжай бить себя в грудь, потому что по-настоящему ты был Сосо только в те минуты, когда курил гашиш.',
 ' Скажи, человек, кто ты такой? Кто тебя прислал? Ты человек или нет? Я или нет? Скажи! А то все опять будут плеваться и говорить, что ты. А это никому не нужно! И вообще, кто ты такой?']